## Intialising Function and Importing Modules

In [1]:
from trainModel import trainModel
from Dataset import ModelDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix, accuracy_score
import pandas as pd
import os
from generateSplits import generateSplits
from torch.utils.data import DataLoader
import torch,gc
import numpy as np
from typing import Literal, Callable, Iterator

In [2]:
DATASET_PATH = r"E:\SRP\SRP-2025-Project\ISPY2_T0_T3_DCE_npz"

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dataset_df = pd.read_excel(r"e:\SRP\ISPY2-Data-Collector\ISPY2-Imaging-Cohort-1-Clinical-Data.xlsx")
dataset_df = dataset_df.set_index("Patient_ID",drop=True)
dataset_df = dataset_df.loc[dataset_df.index.isin([int(os.path.splitext(fname)[0].replace("ISPY2-","")) for fname in os.listdir(DATASET_PATH)]),["HR","HER2","pCR"]]

train_df, test_df = generateSplits(dataset_df,0.2,seed=2008)
skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=2008)

In [3]:
best_params = {'lr': 0.010203264127576701, 'weight_decay': 0.004617603477095672, 'batch_size': 8, 'optimiser_name': 'Adam'}

In [4]:
def evaluate_roc_auc(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2],testKwargs:dict):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for T0_volumes,T3_volumes,mols,labels in test_loader:
            T0_volumes = T0_volumes.to(device)
            T3_volumes = T3_volumes.to(device)
            mols = mols.to(device)
            labels = labels.to(device)
            if combine_timepoints:
                logits = model(torch.cat((T0_volumes,T3_volumes),dim=1),mols,**testKwargs)
            else:
                logits = model(T0_volumes,T3_volumes,mols,**testKwargs)
                
            if out_features == 1:
                score = torch.sigmoid(logits).squeeze()
            elif out_features == 2:
                score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return roc_auc_score(y_true,y_score)

In [5]:
def get_scores(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2],mode:Literal["preNac","both"]="preNac"):
    model.eval()
    y_true=[]
    y_score=[]
    
    with torch.no_grad():
        for T0_volumes,T3_volumes,mols,labels in test_loader:
            T0_volumes = T0_volumes.to(device)
            T3_volumes = T3_volumes.to(device)
            mols = mols.to(device)
            labels = labels.to(device)
            if combine_timepoints:
                logits = model(torch.cat((T0_volumes,T3_volumes),dim=1),mols,mode)
            else:
                logits = model(T0_volumes,T3_volumes,mols,mode)
                
            if out_features == 1:
                score = torch.sigmoid(logits).squeeze()
            elif out_features == 2:
                score = torch.nn.functional.softmax(logits,dim=1)[:,1]
            y_true.extend(labels.cpu().numpy())
            y_score.extend(score.cpu().numpy())
    return y_true,y_score

In [6]:
def score_to_pred(y_true,y_score):
    fpr,tpr,thresholds = roc_curve(y_true,y_score)
    youden_index = tpr-fpr
    best_idx = np.argmax(youden_index)
    best_threshold = thresholds[best_idx]
    y_pred = np.array(y_score)>=best_threshold
    return y_pred

In [7]:
def evaluate_model(model:torch.nn.Module,test_loader:DataLoader,combine_timepoints:bool,out_features:Literal[1,2],mode:Literal["preNac","both"]):
    y_true,y_score = get_scores(model,
                                test_loader,
                                combine_timepoints,
                                out_features,
                                mode)
    y_pred = score_to_pred(y_true,y_score)
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()
    return pd.Series({"ROC AUC":roc_auc_score(y_true,y_score),
                      "Average Precision Score":average_precision_score(y_true,y_score),
                      "Sensitivity / Recall / TPR":(tp/(tp+fn)),
                      "Specificity / TNR":(tn/(tn+fp)),
                      "PPV / Precision":(tp/(tp+fp)),
                      "NPV":(tn/(tn+fn)),
                      "Accuracy":accuracy_score(y_true,y_pred)})

In [8]:
def four_fold_cv_train(Model_class:type[torch.nn.Module],
                       optimiser_class:Callable[[Iterator[torch.nn.Parameter]],torch.optim.Optimizer],
                       class_samples:dict,
                       num_epochs:int,
                       batch_size:int,
                       combine_timepoints:bool,
                       output_features:Literal[1,2],
                       loss_fn:Callable,
                       score_fn:Callable,
                       score_name:str,
                       probs_fn:Callable[[torch.nn.Module,DataLoader,bool,Literal[1,2],Literal["preNac","both"]],tuple],
                       modelKwargs:dict,
                       trainKwargs:dict,
                       testKwargs:dict):
    '''Train model using four-fold cross validation
    
    Parameters
    ----------
    Model_Class : Class to intialise the model
    
    optimiser_class : Class to intialise optimiser
    
    class_samples : Proportion of augmentation for positive and negatives
    
    num_epochs : Number of Epochs
    
    batch_size : Batch Size
    
    combine_timepoints : Whether to combine T0 and T3 into one volume
    
    output_features : Whether the output logits of the model is 1 or 2
    
    loss_fn : Used to calculate loss when training
    
    score_fn : Score used for early stopping
    
    score_name : Score name
    
    probs_fn : Function used to calculate probability of positive class from logits
    
    modelKwargs : kwargs passed to the model during initialisation
    
    trainKwargs : kwargs passed to the model during training
    
    testKwargs : kwargs passed to the model during testing for early stopping'''
    
    combined_metrics = []
    preNac_metrics = []
    
    for train_index,val_index in skf.split(train_df,train_df["pCR"]):
        model = Model_class(**modelKwargs)
        model = model.to(device)
        fold_train_df = train_df.iloc[train_index]
        fold_test_df = train_df.iloc[val_index]
        fold_train_dataset = ModelDataset(fold_train_df,DATASET_PATH,class_samples,loading_bar=True)
        fold_train_loader = DataLoader(fold_train_dataset,batch_size=batch_size,shuffle=True)
        fold_test_dataset = ModelDataset(fold_test_df,DATASET_PATH,loading_bar=True)
        fold_test_loader = DataLoader(fold_test_dataset,batch_size=batch_size)
        model,score = trainModel(model,
                                 train_loader=fold_train_loader,
                                 combine_timepoints=combine_timepoints,
                                 out_features=output_features,
                                 loss_fn = loss_fn,
                                 optimiser=optimiser_class(model.parameters()),
                                 num_epochs=num_epochs,
                                 val_loader=fold_test_loader,
                                 score_fn=score_fn,
                                 score_name=score_name,
                                 patience=num_epochs//4,
                                 trainKwargs=trainKwargs,
                                 testKwargs=testKwargs)
        y_true,y_score = probs_fn(model,fold_test_loader,combine_timepoints,output_features,testKwargs["mode"])
        print(f"Average Precision={(average_precision_score(y_true,y_score)):.4f}")
        combined_metrics.append(evaluate_model(model,
                                               fold_test_loader,
                                               combine_timepoints,
                                               output_features,
                                               mode="both"))
        preNac_metrics.append(evaluate_model(model,
                                               fold_test_loader,
                                               combine_timepoints,
                                               output_features,
                                               mode="preNac"))
        
        del model
        torch.cuda.empty_cache()
        
    return pd.DataFrame({"preNac":sum(preNac_metrics)/len(preNac_metrics),
            "Both":sum(combined_metrics)/len(combined_metrics),}).T.to_markdown()
        

## Model Tests

In [ ]:
from models.CMC_Model.model import Model as CMC_Model

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"both"})
print(four_fold_scores)

'''
'ROC_AUC': 0.5445866499092306, 
'Average Precision': 0.38931456533232567, 
'Sensitivity / Recall / TPR': 0.7067204301075268, 
'Specificity / TNR': 0.4179383116883117, 
'PPV / Precision': 0.4787010990875746, 
'NPV': 0.8056943056943057, 
'Accuracy': 0.5174418604651163
'''


KeyboardInterrupt: 

In [ ]:
from models.CMC_SE.model import Model as CMC_SE_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"both"})
print(four_fold_scores)

'''
'ROC_AUC': 0.561273652422846, 
'Average Precision': 0.4331408019221398, 
'Sensitivity / Recall / TPR': 0.6610215053763441, 
'Specificity / TNR': 0.49261363636363636, 
'PPV / Precision': 0.43590604120695486, 
'NPV': 0.7934704184704184, 
'Accuracy': 0.5524281805745554
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=19.8124
ROC AUC=0.643452380952381
Epoch 1 Done. Average Loss=0.8168
ROC AUC=0.5589285714285714
Epoch 2 Done. Average Loss=0.7230
ROC AUC=0.556547619047619
Epoch 3 Done. Average Loss=0.5936
ROC AUC=0.505357142857143
Epoch 4 Done. Average Loss=0.5848
ROC AUC=0.5154761904761904
Epoch 5 Done. Average Loss=0.5707
ROC AUC=0.4622023809523809
Early stopping triggered at epoch 5. Best ROC AUC=0.643452380952381
Average Precision=0.4947
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=18.8987
ROC AUC=0.5325513196480939
Epoch 1 Done. Average Loss=0.6996
ROC AUC=0.4750733137829912
Epoch 2 Done. Average Loss=0.6752
ROC AUC=0.5120234604105572
Epoch 3 Done. Average Loss=0.6492
ROC AUC=0.5026392961876833
Epoch 4 Done. Average Loss=0.6648
ROC AUC=0.4574780058651026
Epoch 5 Done. Average Loss=0.6361
ROC AUC=0.5079178885630499
Early stopping triggered a

### CMC Model from benchmark paper with Squeeze-Excitation Blocks and Adaptive Average Pooling trained on preNac and postNac
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.665895 |                  0.498972 |                     0.79543  |            0.538474 |          0.484118 | 0.842553 |   0.628625 |
| Both   |  0.593337 |                  0.434265 |                     0.620699 |            0.638231 |          0.484073 | 0.759682 |   0.631737 |

In [ ]:
from models.CMC_SE_with_AvgPool import Model as CMC_SE_with_AP_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_with_AP_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)


#Ignore this
'''
'ROC_AUC': 0.6627363496718336, 
'Average Precision': 0.508258678020139, 
'Sensitivity / Recall / TPR': 0.660752688172043, 
'Specificity / TNR': 0.6693181818181818, 
'PPV / Precision': 0.5264654345102233, 
'NPV': 0.7861817427855164, 
'Accuracy': 0.6667920656634747
'''

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:57<00:00,  4.42it/s]


Dataset initialised with 347 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 86/86 [00:33<00:00,  2.60it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7707
ROC AUC=0.5089285714285714
Epoch 1 Done. Average Loss=0.6902
ROC AUC=0.7107142857142857
Epoch 2 Done. Average Loss=0.6802
ROC AUC=0.6678571428571428
Epoch 3 Done. Average Loss=0.6926
ROC AUC=0.7089285714285714
Epoch 4 Done. Average Loss=0.6957
ROC AUC=0.6369047619047619
Epoch 5 Done. Average Loss=0.6852
ROC AUC=0.6720238095238096
Epoch 6 Done. Average Loss=0.6814
ROC AUC=0.6476190476190475
Early stopping triggered at epoch 6. Best ROC AUC=0.7107142857142857
Average Precision=0.5380


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:21<00:00,  3.13it/s]


...

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 257/257 [00:57<00:00,  4.47it/s]


Dataset initialised with 348 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 85/85 [00:27<00:00,  3.11it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7220
ROC AUC=0.6472727272727273
Epoch 1 Done. Average Loss=0.6961
ROC AUC=0.6515151515151515
Epoch 2 Done. Average Loss=0.6930
ROC AUC=0.6442424242424243
Epoch 3 Done. Average Loss=0.6770
ROC AUC=0.6793939393939394
Epoch 4 Done. Average Loss=0.6750
ROC AUC=0.6587878787878788
Epoch 5 Done. Average Loss=0.6964
ROC AUC=0.6148484848484849
Epoch 6 Done. Average Loss=0.6740
ROC AUC=0.616969696969697
Epoch 7 Done. Average Loss=0.6658
ROC AUC=0.5987878787878789
Epoch 8 Done. Average Loss=0.6516
ROC AUC=0.5824242424242424
Early stopping triggered at epoch 8. Best ROC AUC=0.6793939393939394
Average Precision=0.5042
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.665895 |   

"\n'ROC_AUC': 0.6627363496718336, \n'Average Precision': 0.508258678020139, \n'Sensitivity / Recall / TPR': 0.660752688172043, \n'Specificity / TNR': 0.6693181818181818, \n'PPV / Precision': 0.5264654345102233, \n'NPV': 0.7861817427855164, \n'Accuracy': 0.6667920656634747\n"

### CMC Model from benchmark paper with Squeeze-Excitation Blocks and Adaptive Average Pooling trained on preNac only

In [ ]:
from models.CMC_SE_with_AvgPool import Model as CMC_SE_with_AP_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMC_SE_with_AP_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"preNac"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

### Dual ResNet Model trained on preNac and postNac
|     |  ROC AUC| Average Precision Score| Sensitivity / Recall / TPR| Specificity / TNR| PPV / Precision|      NPV| Accuracy|
| --- | ------- | ---------------------- | ------------------------- | ---------------- | -------------- | ------- | ------- |
preNac| 0.688885|                0.546715|                   0.825269|          0.515666|         0.50344| 0.862092| 0.626094|
Both  | 0.667877|                0.511665|                   0.741935|          0.583523|         0.53107| 0.826510| 0.640800|

In [ ]:
from models.ResNet.model import Model as DualResNetModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"},)
print(four_fold_scores)

'''
         ROC AUC  Average Precision Score  Sensitivity / Recall / TPR  Specificity / TNR  PPV / Precision       NPV  Accuracy
preNac  0.688885                 0.546715                    0.825269           0.515666          0.50344  0.862092  0.626094
Both    0.667877                 0.511665                    0.741935           0.583523          0.53107  0.826510  0.640800
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7154
ROC AUC=0.6279761904761905
Epoch 1 Done. Average Loss=0.6890
ROC AUC=0.6077380952380952
Epoch 2 Done. Average Loss=0.6665
ROC AUC=0.7261904761904763
Epoch 3 Done. Average Loss=0.6606
ROC AUC=0.7238095238095239
Epoch 4 Done. Average Loss=0.6805
ROC AUC=0.7517857142857143
Epoch 5 Done. Average Loss=0.6430
ROC AUC=0.7440476190476191
Epoch 6 Done. Average Loss=0.6601
ROC AUC=0.7363095238095237
Epoch 7 Done. Average Loss=0.6627
ROC AUC=0.7119047619047619
Epoch 8 Done. Average Loss=0.6392
ROC AUC=0.7446428571428572
Epoch 9 Done. Average Loss=0.6536
ROC AUC=0.731845238095238
Early stopping triggered at epoch 9. Best ROC AUC=0.7517857142857143
Average Precision=0.6540
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.6963
ROC AUC=0.5038123167155425
Epoch 1 Done. Average Loss=0.6407
ROC AUC=0.5378299120234604
Epoch 2 Done. Average Los

"\n'ROC_AUC': 0.6992595307917888, \n'Average Precision': 0.50554374036523, \n'Sensitivity / Recall / TPR': 0.7755376344086021, \n'Specificity / TNR': 0.5966720779220779, \n'PPV / Precision': 0.5266954663693795, \n'NPV': 0.8417085295656724, \n'Accuracy': 0.6609439124487004\n"

### Dual ResNet Model trained on preNac only
|     |  ROC AUC| Average Precision Score| Sensitivity / Recall / TPR| Specificity / TNR| PPV / Precision|      NPV| Accuracy|
| --- | ------- | ---------------------- | ------------------------- | ---------------- | -------------- | ------- | ------- |
preNac| 0.686923|                0.562943|                   0.751613|           0.60211|        0.531478| 0.817931| 0.655369|
Both  | 0.686777|                0.562903|                   0.751613|           0.60211|        0.531478| 0.817931| 0.655369|

In [ ]:
from models.ResNet.model import Model as DualResNetModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"preNac"},
                                      testKwargs={"mode":"preNac"},)
print(four_fold_scores)
'''
         ROC AUC  Average Precision Score  Sensitivity / Recall / TPR  Specificity / TNR  PPV / Precision       NPV  Accuracy
preNac  0.686923                 0.562943                    0.751613            0.60211         0.531478  0.817931  0.655369
Both    0.686777                 0.562903                    0.751613            0.60211         0.531478  0.817931  0.655369
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7252
ROC AUC=0.5714285714285714
Epoch 1 Done. Average Loss=0.7093
ROC AUC=0.6410714285714286
Epoch 2 Done. Average Loss=0.6927
ROC AUC=0.6208333333333333
Epoch 3 Done. Average Loss=0.6803
ROC AUC=0.5636904761904762
Epoch 4 Done. Average Loss=0.6914
ROC AUC=0.6642857142857143
Epoch 5 Done. Average Loss=0.6573
ROC AUC=0.6375000000000001
Epoch 6 Done. Average Loss=0.6773
ROC AUC=0.6767857142857143
Epoch 7 Done. Average Loss=0.6880
ROC AUC=0.7166666666666667
Epoch 8 Done. Average Loss=0.6773
ROC AUC=0.6970238095238096
Epoch 9 Done. Average Loss=0.6952
ROC AUC=0.6526785714285714
Epoch 10 Done. Average Loss=0.6617
ROC AUC=0.6482142857142857
Epoch 11 Done. Average Loss=0.6659
ROC AUC=0.6535714285714286
Epoch 12 Done. Average Loss=0.6607
ROC AUC=0.4994047619047619
Early stopping triggered at epoch 12. Best ROC AUC=0.7166666666666667
Average Precision=0.6196
Dataset initialised with 346 entri

### Dual ResNet Model trained on preNac and postNac with SE

In [ ]:
from models.ResNet_SE.ResNet_SE_r_16 import Model as DualResNetSE_r_16_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetSE_r_16_Model,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"both"})
print(four_fold_scores)

'''
'ROC_AUC': 0.6598273634967183, 
'Average Precision': 0.4810334624219552, 
'Sensitivity / Recall / TPR': 0.7448924731182796, 
'Specificity / TNR': 0.592775974025974, 
'PPV / Precision': 0.5024766899766899, 
'NPV': 0.8146424349881797, 
'Accuracy': 0.6462380300957593
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7342
ROC AUC=0.6339285714285715
Epoch 1 Done. Average Loss=0.7007
ROC AUC=0.6702380952380953
Epoch 2 Done. Average Loss=0.6674
ROC AUC=0.6601190476190476
Epoch 3 Done. Average Loss=0.6771
ROC AUC=0.6267857142857143
Epoch 4 Done. Average Loss=0.6952
ROC AUC=0.6202380952380953
Epoch 5 Done. Average Loss=0.6656
ROC AUC=0.6702380952380952
Epoch 6 Done. Average Loss=0.6885
ROC AUC=0.6238095238095238
Early stopping triggered at epoch 6. Best ROC AUC=0.6702380952380953
Average Precision=0.4752
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7160
ROC AUC=0.5824046920821114
Epoch 1 Done. Average Loss=0.6543
ROC AUC=0.509090909090909
Epoch 2 Done. Average Loss=0.6513
ROC AUC=0.5378299120234603
Epoch 3 Done. Average Loss=0.6281
ROC AUC=0.5313782991202346
Epoch 4 Done. Average Loss=0.6558
ROC AUC=0.5020527859237536
Epoch 5 Done. Average Los

In [ ]:
from models.ResNet_SE.ResNet_SE_r_4 import Model as DualResNetSE_r_4_Model
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetSE_r_4_Model,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})
print(four_fold_scores)

'''
'ROC_AUC': 0.691136712749616, 
'Average Precision': 0.5321514416865775, 
'Sensitivity / Recall / TPR': 0.7120967741935483, 
'Specificity / TNR': 0.6651785714285714, 
'PPV / Precision': 0.5407551766436784, 
'NPV': 0.8130799755799756, 
'Accuracy': 0.6814979480164158
'''

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7067
ROC AUC=0.5910714285714286
Epoch 1 Done. Average Loss=0.6985
ROC AUC=0.6339285714285714
Epoch 2 Done. Average Loss=0.6738
ROC AUC=0.6642857142857143
Epoch 3 Done. Average Loss=0.6905
ROC AUC=0.6303571428571428
Epoch 4 Done. Average Loss=0.6419
ROC AUC=0.5630952380952381
Epoch 5 Done. Average Loss=0.6600
ROC AUC=0.7154761904761905
Epoch 6 Done. Average Loss=0.6958
ROC AUC=0.6821428571428572
Epoch 7 Done. Average Loss=0.6550
ROC AUC=0.6136904761904762
Epoch 8 Done. Average Loss=0.6433
ROC AUC=0.738095238095238
Epoch 9 Done. Average Loss=0.6573
ROC AUC=0.7226190476190476
Epoch 10 Done. Average Loss=0.6806
ROC AUC=0.6565476190476192
Epoch 11 Done. Average Loss=0.6716
ROC AUC=0.6607142857142857
Epoch 12 Done. Average Loss=0.6615
ROC AUC=0.6642857142857143
Epoch 13 Done. Average Loss=0.6483
ROC AUC=0.7047619047619048
Early stopping triggered at epoch 13. Best ROC AUC=0.738095238095238

### Dual ResNet Model trained on preNac and postNac with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.694536 |                  0.557168 |                     0.721505 |            0.637662 |          0.518829 | 0.819257 |   0.666587 |
| Both   |  0.64617  |                  0.514268 |                     0.62043  |            0.705357 |          0.542772 | 0.772832 |   0.675547 |

In [ ]:
from models.ResNet.model_with_crossAttentionFusion import Model as DualResNetModel_with_CA
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel_with_CA,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"hidden_dim":128,
                                                   "feedforward_dim":256,
                                                   "fusion_dropout":False},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"},)
print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7100
ROC AUC=0.743452380952381
Epoch 1 Done. Average Loss=0.6997
ROC AUC=0.7208333333333333
Epoch 2 Done. Average Loss=0.6827
ROC AUC=0.7029761904761904
Epoch 3 Done. Average Loss=0.6745
ROC AUC=0.7008928571428571
Epoch 4 Done. Average Loss=0.6771
ROC AUC=0.6821428571428572
Epoch 5 Done. Average Loss=0.6823
ROC AUC=0.6744047619047618
Early stopping triggered at epoch 5. Best ROC AUC=0.743452380952381
Average Precision=0.5458
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7237
ROC AUC=0.4815249266862171
Epoch 1 Done. Average Loss=0.6635
ROC AUC=0.5184750733137831
Epoch 2 Done. Average Loss=0.6438
ROC AUC=0.5026392961876833
Epoch 3 Done. Average Loss=0.6444
ROC AUC=0.5483870967741935
Epoch 4 Done. Average Loss=0.6280
ROC AUC=0.5501466275659823
Epoch 5 Done. Average Loss=0.6309
ROC AUC=0.4997067448680352
Epoch 6 Done. Average Loss

### Dual ResNet Model trained on preNac and postNac with Cross Attention Fusion and Dropout
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |     NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|--------:|-----------:|
| preNac |  0.690598 |                  0.549323 |                     0.636828 |            0.723295 |          0.572192 | 0.78487 |   0.693057 |
| Both   |  0.644887 |                  0.505548 |                     0.670699 |            0.647159 |          0.509851 | 0.78379 |   0.65513  |

In [ ]:
from models.ResNet.model_with_crossAttentionFusionAndDropout import Model as DualResNetModel_with_CADO
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(DualResNetModel_with_CADO,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"hidden_dim":128,
                                                   "feedforward_dim":256,
                                                   "fusion_dropout":True},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"},)
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [00:58<00:00,  4.36it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:19<00:00,  4.50it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7276
ROC AUC=0.7392857142857143
Epoch 1 Done. Average Loss=0.7031
ROC AUC=0.7017857142857143
Epoch 2 Done. Average Loss=0.6906
ROC AUC=0.6678571428571428
Epoch 3 Done. Average Loss=0.6685
ROC AUC=0.6898809523809524
Epoch 4 Done. Average Loss=0.6571
ROC AUC=0.6904761904761905
Epoch 5 Done. Average Loss=0.6926
ROC AUC=0.7404761904761905
Epoch 6 Done. Average Loss=0.6701
ROC AUC=0.7193452380952382
Epoch 7 Done. Average Loss=0.6688
ROC AUC=0.6863095238095238
Epoch 8 Done. Average Loss=0.6646
ROC AUC=0.674404761904762
Epoch 9 Done. Average Loss=0.6633
ROC AUC=0.7306547619047619
Epoch 10 Done. Average Loss=0.6602
ROC AUC=0.6758928571428572
Early stopping triggered at epoch 10. Best ROC AUC=0.7404761904761905
Average Precision=0.6008


100%|████████████████████████████████████████████████████████████████████| 256/256 [01:02<00:00,  4.09it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7332
ROC AUC=0.6624242424242425
Epoch 1 Done. Average Loss=0.6876
ROC AUC=0.5775757575757576
Epoch 2 Done. Average Loss=0.6867
ROC AUC=0.7115151515151515
Epoch 3 Done. Average Loss=0.6738
ROC AUC=0.6666666666666666
Epoch 4 Done. Average Loss=0.6839
ROC AUC=0.6593939393939394
Epoch 5 Done. Average Loss=0.6659
ROC AUC=0.6630303030303031
Epoch 6 Done. Average Loss=0.6671
ROC AUC=0.7115151515151515
Epoch 7 Done. Average Loss=0.6670
ROC AUC=0.66
Early stopping triggered at epoch 7. Best ROC AUC=0.7115151515151515
Average Precision=0.5819


100%|████████████████████████████████████████████████████████████████████| 257/257 [00:58<00:00,  4.43it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:26<00:00,  3.16it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7970
ROC AUC=0.6993939393939395
Epoch 1 Done. Average Loss=0.6791
ROC AUC=0.7375757575757576
Epoch 2 Done. Average Loss=0.6732
ROC AUC=0.7496969696969698
Epoch 3 Done. Average Loss=0.6757
ROC AUC=0.7133333333333334
Epoch 4 Done. Average Loss=0.6718
ROC AUC=0.7387878787878788
Epoch 5 Done. Average Loss=0.6674
ROC AUC=0.7030303030303031
Epoch 6 Done. Average Loss=0.6843
ROC AUC=0.6727272727272727
Epoch 7 Done. Average Loss=0.6665
ROC AUC=0.6787878787878787
Early stopping triggered at epoch 7. Best ROC AUC=0.7496969696969698
Average Precision=0.6221
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |     NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|--------:|-----------:|
| preNac |  0.690598 |                  0.549323 |                     0.636828 |      

### Conv Mixer 128/4 trained on preNac and postNac
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.681448 |                  0.497225 |                     0.851075 |            0.515828 |          0.498676 | 0.858586 |   0.63485  |
| Both   |  0.618113 |                  0.447283 |                     0.802688 |            0.470617 |          0.45785  | 0.831653 |   0.587688 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":4,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":64},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)


Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7183
ROC AUC=0.6386904761904761
Epoch 1 Done. Average Loss=0.6436
ROC AUC=0.7029761904761905
Epoch 2 Done. Average Loss=0.6103
ROC AUC=0.6886904761904762
Epoch 3 Done. Average Loss=0.6308
ROC AUC=0.5148809523809523
Epoch 4 Done. Average Loss=0.6105
ROC AUC=0.5523809523809524
Epoch 5 Done. Average Loss=0.5685
ROC AUC=0.675
Epoch 6 Done. Average Loss=0.5575
ROC AUC=0.6178571428571429
Early stopping triggered at epoch 6. Best ROC AUC=0.7029761904761905
Average Precision=0.4818
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.6752
ROC AUC=0.5882697947214076
Epoch 1 Done. Average Loss=0.6249
ROC AUC=0.5255131964809384
Epoch 2 Done. Average Loss=0.6175
ROC AUC=0.5501466275659824
Epoch 3 Done. Average Loss=0.6096
ROC AUC=0.501466275659824
Epoch 4 Done. Average Loss=0.5772
ROC AUC=0.5595307917888563
Epoch 5 Done. Average Loss=0.5909
ROC 

### Conv Mixer 256/6 trained on preNac and postNac
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.678469 |                  0.544371 |                     0.702957 |            0.637662 |          0.524984 | 0.801245 |    0.66091 |
| Both   |  0.612599 |                  0.478796 |                     0.540054 |            0.733117 |          0.551455 | 0.761019 |    0.66368 

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":256,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7870
ROC AUC=0.594047619047619
Epoch 1 Done. Average Loss=0.7332
ROC AUC=0.5452380952380953
Epoch 2 Done. Average Loss=0.6439
ROC AUC=0.6761904761904762
Epoch 3 Done. Average Loss=0.6396
ROC AUC=0.7250000000000001
Epoch 4 Done. Average Loss=0.5938
ROC AUC=0.593452380952381
Epoch 5 Done. Average Loss=0.6657
ROC AUC=0.7101190476190475
Epoch 6 Done. Average Loss=0.6013
ROC AUC=0.6845238095238095
Epoch 7 Done. Average Loss=0.5995
ROC AUC=0.6976190476190477
Epoch 8 Done. Average Loss=0.6172
ROC AUC=0.6607142857142858
Early stopping triggered at epoch 8. Best ROC AUC=0.7250000000000001
Average Precision=0.5731
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7272
ROC AUC=0.5079178885630499
Epoch 1 Done. Average Loss=0.6631
ROC AUC=0.49266862170087977
Epoch 2 Done. Average Loss=0.6442
ROC AUC=0.5519061583577712
Epoch 3 Done. Average Los

### Conv Mixer 128/6 trained on preNac and postNac, kernelsize = (1,9,9), patchsize = (8,8,8)
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.695554 |                  0.558633 |                     0.785215 |            0.579383 |          0.51173  | 0.829545 |   0.652326 |
| Both   |  0.617869 |                  0.471568 |                     0.8      |            0.464286 |          0.482102 | 0.876818 |   0.584918 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7616
ROC AUC=0.5886904761904762
Epoch 1 Done. Average Loss=0.6953
ROC AUC=0.6351190476190476
Epoch 2 Done. Average Loss=0.6074
ROC AUC=0.7017857142857142
Epoch 3 Done. Average Loss=0.6474
ROC AUC=0.6720238095238095
Epoch 4 Done. Average Loss=0.6196
ROC AUC=0.6821428571428572
Epoch 5 Done. Average Loss=0.6155
ROC AUC=0.6595238095238095
Epoch 6 Done. Average Loss=0.6039
ROC AUC=0.6160714285714285
Epoch 7 Done. Average Loss=0.6392
ROC AUC=0.5910714285714286
Early stopping triggered at epoch 7. Best ROC AUC=0.7017857142857142
Average Precision=0.5746
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7500
ROC AUC=0.6052785923753665
Epoch 1 Done. Average Loss=0.6145
ROC AUC=0.4656891495601173
Epoch 2 Done. Average Loss=0.6329
ROC AUC=0.5378299120234604
Epoch 3 Done. Average Loss=0.6561
ROC AUC=0.5208211143695015
Epoch 4 Done. Average Lo

### Conv Mixer 128/6 trained on preNac only
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.645287 |                  0.51019  |                      0.79543 |            0.502192 |          0.463588 | 0.833445 |   0.605369 |
| Both   |  0.645436 |                  0.510238 |                      0.79543 |            0.502192 |          0.463588 | 0.833445 |   0.605369 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"preNac"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7151
ROC AUC=0.6053571428571429
Epoch 1 Done. Average Loss=0.7149
ROC AUC=0.6702380952380953
Epoch 2 Done. Average Loss=0.6808
ROC AUC=0.5136904761904761
Epoch 3 Done. Average Loss=0.6595
ROC AUC=0.5672619047619049
Epoch 4 Done. Average Loss=0.6617
ROC AUC=0.6029761904761904
Epoch 5 Done. Average Loss=0.6878
ROC AUC=0.5994047619047619
Epoch 6 Done. Average Loss=0.6576
ROC AUC=0.6636904761904762
Early stopping triggered at epoch 6. Best ROC AUC=0.6702380952380953
Average Precision=0.5699
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7579
ROC AUC=0.509090909090909
Epoch 1 Done. Average Loss=0.6276
ROC AUC=0.4868035190615836
Epoch 2 Done. Average Loss=0.6455
ROC AUC=0.4686217008797654
Epoch 3 Done. Average Loss=0.6301
ROC AUC=0.4633431085043988
Epoch 4 Done. Average Loss=0.6135
ROC AUC=0.42815249266862176
Epoch 5 Done. Average Lo

### Conv Mixer 256/6 trained on preNac and postNac, patchsize = (16,16,16)
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.665124 |                  0.553026 |                     0.563172 |            0.751055 |          0.575326 | 0.765601 |   0.6842   |
| Both   |  0.596229 |                  0.490814 |                     0.481452 |            0.76112  |          0.560188 | 0.738757 |   0.660841 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":256,
                                                   "depth":6,
                                                   "kernel_size":(1,5,5),
                                                   "patch_size":(16,16,16),
                                                   "mol_dim":64,
                                                   "hidden_fusion_dim":256},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.8315
ROC AUC=0.46904761904761905
Epoch 1 Done. Average Loss=0.7780
ROC AUC=0.6696428571428571
Epoch 2 Done. Average Loss=0.7439
ROC AUC=0.5869047619047619
Epoch 3 Done. Average Loss=0.6924
ROC AUC=0.5458333333333333
Epoch 4 Done. Average Loss=0.6359
ROC AUC=0.7041666666666666
Epoch 5 Done. Average Loss=0.6949
ROC AUC=0.5827380952380953
Epoch 6 Done. Average Loss=0.6594
ROC AUC=0.6422619047619048
Epoch 7 Done. Average Loss=0.6567
ROC AUC=0.6386904761904761
Epoch 8 Done. Average Loss=0.6111
ROC AUC=0.7125
Epoch 9 Done. Average Loss=0.5835
ROC AUC=0.6732142857142857
Epoch 10 Done. Average Loss=0.5972
ROC AUC=0.6988095238095238
Epoch 11 Done. Average Loss=0.6326
ROC AUC=0.6351190476190477
Epoch 12 Done. Average Loss=0.5781
ROC AUC=0.6648809523809525
Epoch 13 Done. Average Loss=0.5766
ROC AUC=0.6488095238095238
Early stopping triggered at epoch 13. Best ROC AUC=0.7125
Average Precision=0.

### Conv Mixer 128/6 trained on preNac and postNac, kernelsize = (1,11,11), patchsize = (8,8,8)
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.693498 |                  0.55983  |                     0.67043  |            0.736851 |          0.602927 | 0.809555 |   0.713509 |
| Both   |  0.619879 |                  0.485253 |                     0.630376 |            0.641802 |          0.510195 | 0.789632 |   0.637209 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,11,11),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7805
ROC AUC=0.5729166666666667
Epoch 1 Done. Average Loss=0.6554
ROC AUC=0.7059523809523809
Epoch 2 Done. Average Loss=0.6322
ROC AUC=0.6738095238095237
Epoch 3 Done. Average Loss=0.6642
ROC AUC=0.731547619047619
Epoch 4 Done. Average Loss=0.6151
ROC AUC=0.6666666666666667
Epoch 5 Done. Average Loss=0.6283
ROC AUC=0.6880952380952381
Epoch 6 Done. Average Loss=0.6288
ROC AUC=0.6660714285714286
Epoch 7 Done. Average Loss=0.6097
ROC AUC=0.6660714285714285
Epoch 8 Done. Average Loss=0.6034
ROC AUC=0.6488095238095237
Early stopping triggered at epoch 8. Best ROC AUC=0.731547619047619
Average Precision=0.6294
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7280
ROC AUC=0.4639296187683285
Epoch 1 Done. Average Loss=0.6741
ROC AUC=0.5372434017595308
Epoch 2 Done. Average Loss=0.6391
ROC AUC=0.4727272727272728
Epoch 3 Done. Average Loss

### Conv Mixer 64/6 trained on preNac and postNac, kernelsize = (1,9,9), patchsize = (8,8,8)
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.645369 |                  0.480846 |                     0.652688 |            0.687825 |          0.548209 | 0.782817 |   0.675684 |
| Both   |  0.62469  |                  0.450277 |                     0.795161 |            0.524675 |          0.476318 | 0.8361   |   0.619904 |

In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":64,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":16,
                                                   "hidden_fusion_dim":64},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7369
ROC AUC=0.5994047619047619
Epoch 1 Done. Average Loss=0.6511
ROC AUC=0.6499999999999999
Epoch 2 Done. Average Loss=0.6298
ROC AUC=0.6357142857142858
Epoch 3 Done. Average Loss=0.6450
ROC AUC=0.6083333333333334
Epoch 4 Done. Average Loss=0.5991
ROC AUC=0.5666666666666667
Epoch 5 Done. Average Loss=0.6170
ROC AUC=0.6428571428571429
Epoch 6 Done. Average Loss=0.6038
ROC AUC=0.6446428571428571
Early stopping triggered at epoch 6. Best ROC AUC=0.6499999999999999
Average Precision=0.4341
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7111
ROC AUC=0.49618768328445745
Epoch 1 Done. Average Loss=0.6139
ROC AUC=0.5466275659824047
Epoch 2 Done. Average Loss=0.6474
ROC AUC=0.5272727272727272
Epoch 3 Done. Average Loss=0.6166
ROC AUC=0.5225806451612902
Epoch 4 Done. Average Loss=0.6034
ROC AUC=0.46392961876832844
Epoch 5 Done. Average 

### Conv Mixer 128/8 trained on preNac and postNac, kernelsize = (1,9,9), patchsize = (8,8,8)
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.67972  |                  0.512266 |                     0.792204 |            0.5375   |           0.49787 | 0.840326 |   0.628762 |
| Both   |  0.565595 |                  0.413064 |                     0.708602 |            0.515341 |           0.47408 | 0.818523 |   0.58485  |


In [ ]:
from models.ConvMixer import Model as ConvMixerModel
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerModel,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7571
ROC AUC=0.5684523809523809
Epoch 1 Done. Average Loss=0.6659
ROC AUC=0.7077380952380953
Epoch 2 Done. Average Loss=0.6573
ROC AUC=0.6726190476190477
Epoch 3 Done. Average Loss=0.6376
ROC AUC=0.6190476190476191
Epoch 4 Done. Average Loss=0.6242
ROC AUC=0.6505952380952381
Epoch 5 Done. Average Loss=0.6131
ROC AUC=0.6125
Epoch 6 Done. Average Loss=0.5704
ROC AUC=0.6654761904761906
Early stopping triggered at epoch 6. Best ROC AUC=0.7077380952380953
Average Precision=0.5229
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7274
ROC AUC=0.509090909090909
Epoch 1 Done. Average Loss=0.6208
ROC AUC=0.5255131964809384
Epoch 2 Done. Average Loss=0.6244
ROC AUC=0.473900293255132
Epoch 3 Done. Average Loss=0.6239
ROC AUC=0.5079178885630499
Epoch 4 Done. Average Loss=0.6087
ROC AUC=0.5032258064516129
Epoch 5 Done. Average Loss=0.5958
ROC 

### Conv Mixer 128/6 trained on preNac and postNac, kernelsize = (1,9,9), patchsize = (8,8,8) with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.696357 |                  0.544979 |                     0.867473 |            0.479383 |          0.480868 | 0.867984 |   0.617134 |
| Both   |  0.643487 |                  0.492605 |                     0.718548 |            0.583117 |          0.497264 | 0.788119 |   0.631703 |

In [ ]:
from models.ConvMixerWithCAFusion import Model as ConvMixerWithCAFusion
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerWithCAFusion,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"dim":128,
                                                   "depth":6,
                                                   "kernel_size":(1,9,9),
                                                   "patch_size":(8,8,8),
                                                   "mol_dim":32,
                                                   "CA_feedforward_dim":256,
                                                   "hidden_fusion_dim":64},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:09<00:00,  3.70it/s]


Dataset initialised with 347 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 86/86 [00:29<00:00,  2.87it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7394
ROC AUC=0.7196428571428571
Epoch 1 Done. Average Loss=0.6777
ROC AUC=0.6541666666666667
Epoch 2 Done. Average Loss=0.6718
ROC AUC=0.6154761904761905
Epoch 3 Done. Average Loss=0.6734
ROC AUC=0.6666666666666667
Epoch 4 Done. Average Loss=0.6687
ROC AUC=0.6571428571428571
Epoch 5 Done. Average Loss=0.6717
ROC AUC=0.6351190476190477
Early stopping triggered at epoch 5. Best ROC AUC=0.7196428571428571
Average Precision=0.5230


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:31<00:00,  2.79it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7446
ROC AUC=0.7121212121212122
Epoch 1 Done. Average Loss=0.7093
ROC AUC=0.6733333333333333
Epoch 2 Done. Average Loss=0.6626
ROC AUC=0.6593939393939394
Epoch 3 Done. Average Loss=0.6785
ROC AUC=0.7018181818181819
Epoch 4 Done. Average Loss=0.6531
ROC AUC=0.6751515151515152
Epoch 5 Done. Average Loss=0.6454
ROC AUC=0.6357575757575757
Early stopping triggered at epoch 5. Best ROC AUC=0.7121212121212122
Average Precision=0.5962


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 257/257 [01:02<00:00,  4.09it/s]


Dataset initialised with 348 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 85/85 [00:19<00:00,  4.45it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7298
ROC AUC=0.7109090909090909
Epoch 1 Done. Average Loss=0.7053
ROC AUC=0.6812121212121213
Epoch 2 Done. Average Loss=0.6785
ROC AUC=0.7078787878787879
Epoch 3 Done. Average Loss=0.6715
ROC AUC=0.7139393939393939
Epoch 4 Done. Average Loss=0.6468
ROC AUC=0.6484848484848484
Epoch 5 Done. Average Loss=0.6778
ROC AUC=0.713939393939394
Epoch 6 Done. Average Loss=0.6628
ROC AUC=0.6636363636363637
Epoch 7 Done. Average Loss=0.6556
ROC AUC=0.6909090909090909
Epoch 8 Done. Average Loss=0.6690
ROC AUC=0.7636363636363637
Epoch 9 Done. Average Loss=0.6800
ROC AUC=0.7424242424242424
Epoch 10 Done. Average Loss=0.6481
ROC AUC=0.6393939393939394
Epoch 11 Done. Average Loss=0.6556
ROC AUC=0.6260606060606061
Epoch 12 Done. Average Loss=0.6655
ROC AUC=0.676969696969697
Epoch 13 Done. Average Loss=0.6776
ROC AUC=0.696969696969697
Early stopping triggered at epoch 13. Best ROC AUC=0.7636363636363637
Average Precision=0.6504
|        |   

### ViT 
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.669382 |                  0.493065 |                     0.745968 |            0.543344 |          0.467905 | 0.811645 |   0.614124 |
| Both   |  0.601778 |                  0.439365 |                     0.612634 |            0.624351 |          0.467768 | 0.755798 |   0.620109 |

In [ ]:
from models.ViT import Model as ViT
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ViT,
                                      optimiser_class,
                                      {0:1,1:2},
                                      num_epochs=20,
                                      batch_size=8,
                                      combine_timepoints=False,
                                      output_features=1,
                                      loss_fn=loss_fn,
                                      score_fn=evaluate_roc_auc,
                                      score_name="ROC AUC",
                                      probs_fn=get_scores,
                                      modelKwargs={"image_size":(16,128,128),
                                                   "hidden_dim":128,
                                                   "patch_size":(8,8,8),
                                                   "num_heads":4,
                                                   "feedforward_dim":256,
                                                   "num_blocks":6,
                                                   "mol_dim":32,
                                                   "hidden_fusion_dim":128},
                                      trainKwargs={"mode":"both"},
                                      testKwargs={"mode":"preNac"})

print(four_fold_scores)


Dataset initialised with 347 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.8260
ROC AUC=0.48452380952380947
Epoch 1 Done. Average Loss=0.7045
ROC AUC=0.6041666666666667
Epoch 2 Done. Average Loss=0.6949
ROC AUC=0.7386904761904762
Epoch 3 Done. Average Loss=0.6974
ROC AUC=0.6601190476190476
Epoch 4 Done. Average Loss=0.7026
ROC AUC=0.7375
Epoch 5 Done. Average Loss=0.6976
ROC AUC=0.7401785714285714
Epoch 6 Done. Average Loss=0.6943
ROC AUC=0.7386904761904762
Epoch 7 Done. Average Loss=0.6947
ROC AUC=0.7386904761904762
Epoch 8 Done. Average Loss=0.6940
ROC AUC=0.7386904761904762
Epoch 9 Done. Average Loss=0.6949
ROC AUC=0.7386904761904762
Epoch 10 Done. Average Loss=0.6975
ROC AUC=0.6601190476190476
Early stopping triggered at epoch 10. Best ROC AUC=0.7401785714285714
Average Precision=0.5605
Dataset initialised with 346 entries.
Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7827
ROC AUC=0.5217008797653959
Epoch 1 Done. Average Loss=0.7054

In [ ]:
gc.collect()

80

### CMCGCN
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.692793 |                  0.572015 |                     0.643548 |            0.728247 |          0.597892 | 0.788888 |   0.699077 |
| Both   |  0.672032 |                  0.561983 |                     0.785215 |            0.556494 |          0.503754 | 0.822674 |   0.637688 |

In [ ]:
from models.GCNs.CMCGCN import GNNpCRModel as CMCGCN_Model

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.CrossEntropyLoss()
four_fold_scores = four_fold_cv_train(CMCGCN_Model,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=True,
                                             output_features=2,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [00:55<00:00,  4.57it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:18<00:00,  4.56it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.8052
ROC AUC=0.6440476190476191
Epoch 1 Done. Average Loss=0.6947
ROC AUC=0.7172619047619048
Epoch 2 Done. Average Loss=0.6857
ROC AUC=0.6958333333333334
Epoch 3 Done. Average Loss=0.6810
ROC AUC=0.7505952380952381
Epoch 4 Done. Average Loss=0.6833
ROC AUC=0.7678571428571429
Epoch 5 Done. Average Loss=0.6749
ROC AUC=0.7779761904761905
Epoch 6 Done. Average Loss=0.6846
ROC AUC=0.7663690476190477
Epoch 7 Done. Average Loss=0.6825
ROC AUC=0.7511904761904762
Epoch 8 Done. Average Loss=0.6893
ROC AUC=0.7065476190476191
Epoch 9 Done. Average Loss=0.6847
ROC AUC=0.693154761904762
Epoch 10 Done. Average Loss=0.6933
ROC AUC=0.7470238095238095
Early stopping triggered at epoch 10. Best ROC AUC=0.7779761904761905
Average Precision=0.6625


100%|████████████████████████████████████████████████████████████████████| 256/256 [01:59<00:00,  2.14it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.8277
ROC AUC=0.693939393939394
Epoch 1 Done. Average Loss=0.6872
ROC AUC=0.6945454545454545
Epoch 2 Done. Average Loss=0.6842
ROC AUC=0.7181818181818181
Epoch 3 Done. Average Loss=0.6912
ROC AUC=0.7321212121212122
Epoch 4 Done. Average Loss=0.6584
ROC AUC=0.6921212121212121
Epoch 5 Done. Average Loss=0.6936
ROC AUC=0.6624242424242425
Epoch 6 Done. Average Loss=0.6775
ROC AUC=0.6448484848484849
Epoch 7 Done. Average Loss=0.6895
ROC AUC=0.6854545454545454
Epoch 8 Done. Average Loss=0.6741
ROC AUC=0.6503030303030304
Early stopping triggered at epoch 8. Best ROC AUC=0.7321212121212122
Average Precision=0.6235


100%|████████████████████████████████████████████████████████████████████| 257/257 [01:28<00:00,  2.89it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:26<00:00,  3.23it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.9481
ROC AUC=0.6987878787878788
Epoch 1 Done. Average Loss=0.6730
ROC AUC=0.7151515151515151
Epoch 2 Done. Average Loss=0.6928
ROC AUC=0.7296969696969696
Epoch 3 Done. Average Loss=0.6834
ROC AUC=0.6818181818181818
Epoch 4 Done. Average Loss=0.7126
ROC AUC=0.6654545454545454
Epoch 5 Done. Average Loss=0.6927
ROC AUC=0.7212121212121212
Epoch 6 Done. Average Loss=0.6797
ROC AUC=0.6806060606060607
Epoch 7 Done. Average Loss=0.6930
ROC AUC=0.706060606060606
Early stopping triggered at epoch 7. Best ROC AUC=0.7296969696969696
Average Precision=0.6050
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.692793 |                  0.572015 |                     0.643548 |     

### ConvMixerGCN
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.643903 |                  0.467594 |                     0.662366 |            0.63401  |          0.51229  | 0.788008 |   0.643468 |
| Both   |  0.556062 |                  0.426452 |                     0.711559 |            0.461688 |          0.427857 | 0.753571 |   0.549692 |

In [ ]:
from models.GCNs.ConvMixerGCN import Model as ConvMixerGCNModel

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerGCNModel,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "patch_size":(8,8,8),
                                                            "mol_dim":32,
                                                            "hidden_fusion_dim":128,
                                                            "gnn_hidden":64,
                                                            "gnn_out":64},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [01:51<00:00,  2.29it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:34<00:00,  2.50it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.6953
ROC AUC=0.6946428571428571
Epoch 1 Done. Average Loss=0.6882
ROC AUC=0.5410714285714285
Epoch 2 Done. Average Loss=0.6533
ROC AUC=0.6886904761904762
Epoch 3 Done. Average Loss=0.6601
ROC AUC=0.6476190476190476
Epoch 4 Done. Average Loss=0.6285
ROC AUC=0.506547619047619
Epoch 5 Done. Average Loss=0.6168
ROC AUC=0.5166666666666667
Early stopping triggered at epoch 5. Best ROC AUC=0.6946428571428571
Average Precision=0.4699


100%|████████████████████████████████████████████████████████████████████| 256/256 [02:00<00:00,  2.12it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7389
ROC AUC=0.6109090909090908
Epoch 1 Done. Average Loss=0.6684
ROC AUC=0.6169696969696971
Epoch 2 Done. Average Loss=0.6468
ROC AUC=0.6581818181818182
Epoch 3 Done. Average Loss=0.6437
ROC AUC=0.5957575757575757
Epoch 4 Done. Average Loss=0.6317
ROC AUC=0.676060606060606
Epoch 5 Done. Average Loss=0.6182
ROC AUC=0.5890909090909091
Epoch 6 Done. Average Loss=0.6363
ROC AUC=0.5345454545454545
Epoch 7 Done. Average Loss=0.5873
ROC AUC=0.6518181818181819
Epoch 8 Done. Average Loss=0.6090
ROC AUC=0.636969696969697
Epoch 9 Done. Average Loss=0.5982
ROC AUC=0.663030303030303
Early stopping triggered at epoch 9. Best ROC AUC=0.676060606060606
Average Precision=0.5328


100%|████████████████████████████████████████████████████████████████████| 257/257 [01:18<00:00,  3.29it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:21<00:00,  3.93it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7163
ROC AUC=0.5121212121212121
Epoch 1 Done. Average Loss=0.6161
ROC AUC=0.6793939393939394
Epoch 2 Done. Average Loss=0.6477
ROC AUC=0.5987878787878789
Epoch 3 Done. Average Loss=0.6174
ROC AUC=0.6521212121212122
Epoch 4 Done. Average Loss=0.6278
ROC AUC=0.6242424242424243
Epoch 5 Done. Average Loss=0.6193
ROC AUC=0.5987878787878788
Epoch 6 Done. Average Loss=0.6327
ROC AUC=0.5490909090909091
Early stopping triggered at epoch 6. Best ROC AUC=0.6793939393939394
Average Precision=0.4820
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.643903 |                  0.467594 |                     0.662366 |            0.63401  |          0.51229  | 0.788008 |   0.643468 

### ConvMixerGCNv2
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.632287 |                  0.487066 |                     0.661022 |            0.652516 |          0.5315   | 0.783932 |   0.655335 |
| Both   |  0.58755  |                  0.432049 |                     0.841935 |            0.39375  |          0.440806 | 0.828427 |   0.553078 |

In [ ]:
from models.GCNs.ConvMixerGCNv2 import Model as ConvMixerGCNModelv2

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerGCNModelv2,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "patch_size":(8,8,8),
                                                            "mol_dim":32,
                                                            "hidden_fusion_dim":128,
                                                            "gnn_hidden_dim":64,
                                                            "gnn_out_dim":64},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [01:00<00:00,  4.24it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:23<00:00,  3.68it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7607
ROC AUC=0.5553571428571429
Epoch 1 Done. Average Loss=0.7025
ROC AUC=0.6386904761904761
Epoch 2 Done. Average Loss=0.7009
ROC AUC=0.525
Epoch 3 Done. Average Loss=0.6351
ROC AUC=0.5678571428571428
Epoch 4 Done. Average Loss=0.6655
ROC AUC=0.5208333333333333
Epoch 5 Done. Average Loss=0.6413
ROC AUC=0.6375000000000001
Epoch 6 Done. Average Loss=0.6021
ROC AUC=0.5523809523809524
Early stopping triggered at epoch 6. Best ROC AUC=0.6386904761904761
Average Precision=0.4218


100%|████████████████████████████████████████████████████████████████████| 256/256 [01:06<00:00,  3.83it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7273
ROC AUC=0.6254545454545455
Epoch 1 Done. Average Loss=0.7140
ROC AUC=0.49939393939393933
Epoch 2 Done. Average Loss=0.7168
ROC AUC=0.64
Epoch 3 Done. Average Loss=0.6642
ROC AUC=0.6193939393939394
Epoch 4 Done. Average Loss=0.6376
ROC AUC=0.6357575757575757
Epoch 5 Done. Average Loss=0.6430
ROC AUC=0.5909090909090909
Epoch 6 Done. Average Loss=0.6599
ROC AUC=0.6951515151515152
Epoch 7 Done. Average Loss=0.6208
ROC AUC=0.6854545454545454
Epoch 8 Done. Average Loss=0.6274
ROC AUC=0.6933333333333334
Epoch 9 Done. Average Loss=0.6630
ROC AUC=0.6690909090909092
Epoch 10 Done. Average Loss=0.6602
ROC AUC=0.6321212121212121
Epoch 11 Done. Average Loss=0.6155
ROC AUC=0.673939393939394
Early stopping triggered at epoch 11. Best ROC AUC=0.6951515151515152
Average Precision=0.5791


100%|████████████████████████████████████████████████████████████████████| 257/257 [00:59<00:00,  4.29it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:24<00:00,  3.42it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7200
ROC AUC=0.5345454545454544
Epoch 1 Done. Average Loss=0.6971
ROC AUC=0.5230303030303031
Epoch 2 Done. Average Loss=0.7017
ROC AUC=0.5787878787878789
Epoch 3 Done. Average Loss=0.6894
ROC AUC=0.6181818181818182
Epoch 4 Done. Average Loss=0.6730
ROC AUC=0.5024242424242424
Epoch 5 Done. Average Loss=0.7165
ROC AUC=0.4509090909090909
Epoch 6 Done. Average Loss=0.6983
ROC AUC=0.5115151515151515
Epoch 7 Done. Average Loss=0.6705
ROC AUC=0.566060606060606
Epoch 8 Done. Average Loss=0.6538
ROC AUC=0.5763636363636364
Early stopping triggered at epoch 8. Best ROC AUC=0.6181818181818182
Average Precision=0.5294
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.632287 |   

### ConvMixerGCNv2 with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.691984 |                  0.555676 |                     0.695968 |            0.633604 |          0.505643 | 0.798951 |   0.655096 |
| Both   |  0.660701 |                  0.531115 |                     0.778495 |            0.551948 |          0.484854 | 0.826471 |   0.631601 |

In [ ]:
from models.GCNs.ConvMixerGCNv2withCrossAttentionFusion import Model as ConvMixerGCNModelv2withCAFusion

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerGCNModelv2withCAFusion,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "patch_size":(8,8,8),
                                                            "mol_dim":32,
                                                            "feedforward_dim":128,
                                                            "hidden_fusion_dim":128,
                                                            "gnn_hidden_dim":64,
                                                            "gnn_out_dim":64},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [01:19<00:00,  3.21it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:20<00:00,  4.23it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7364
ROC AUC=0.7345238095238096
Epoch 1 Done. Average Loss=0.6826
ROC AUC=0.75
Epoch 2 Done. Average Loss=0.7015
ROC AUC=0.6190476190476191
Epoch 3 Done. Average Loss=0.6787
ROC AUC=0.7517857142857143
Epoch 4 Done. Average Loss=0.6637
ROC AUC=0.6363095238095239
Epoch 5 Done. Average Loss=0.6789
ROC AUC=0.6732142857142858
Epoch 6 Done. Average Loss=0.6717
ROC AUC=0.7476190476190476
Epoch 7 Done. Average Loss=0.6576
ROC AUC=0.718452380952381
Epoch 8 Done. Average Loss=0.6563
ROC AUC=0.6964285714285714
Early stopping triggered at epoch 8. Best ROC AUC=0.7517857142857143
Average Precision=0.6196


100%|████████████████████████████████████████████████████████████████████| 256/256 [03:13<00:00,  1.32it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7515
ROC AUC=0.7206060606060606
Epoch 1 Done. Average Loss=0.6843
ROC AUC=0.6703030303030304
Epoch 2 Done. Average Loss=0.6788
ROC AUC=0.6896969696969697
Epoch 3 Done. Average Loss=0.6757
ROC AUC=0.6854545454545454
Epoch 4 Done. Average Loss=0.6652
ROC AUC=0.6957575757575757
Epoch 5 Done. Average Loss=0.6787
ROC AUC=0.6981818181818182
Early stopping triggered at epoch 5. Best ROC AUC=0.7206060606060606
Average Precision=0.5922


100%|████████████████████████████████████████████████████████████████████| 257/257 [01:46<00:00,  2.42it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:25<00:00,  3.38it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7326
ROC AUC=0.6527272727272727
Epoch 1 Done. Average Loss=0.7084
ROC AUC=0.6987878787878788
Epoch 2 Done. Average Loss=0.6697
ROC AUC=0.6503030303030303
Epoch 3 Done. Average Loss=0.6964
ROC AUC=0.6060606060606061
Epoch 4 Done. Average Loss=0.6540
ROC AUC=0.7012121212121213
Epoch 5 Done. Average Loss=0.6754
ROC AUC=0.7436363636363637
Epoch 6 Done. Average Loss=0.6705
ROC AUC=0.7036363636363636
Epoch 7 Done. Average Loss=0.6615
ROC AUC=0.7187878787878789
Epoch 8 Done. Average Loss=0.6686
ROC AUC=0.7163636363636364
Epoch 9 Done. Average Loss=0.6656
ROC AUC=0.7175757575757576
Epoch 10 Done. Average Loss=0.6687
ROC AUC=0.7227272727272728
Early stopping triggered at epoch 10. Best ROC AUC=0.7436363636363637
Average Precision=0.6036
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------

### ConvMixer6KNNGCN with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.692916 |                  0.565139 |                     0.713441 |            0.642938 |          0.53278  | 0.820987 |   0.666621 |
| Both   |  0.668825 |                  0.529188 |                     0.704839 |            0.647321 |          0.516311 | 0.812006 |   0.666758 |

In [ ]:
from models.GCNs.ConvMixer6KNNGCNwithCrossAttentionFusion import Model as ConvMixer6KNNGCNwithCAFusion
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixer6KNNGCNwithCAFusion,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "patch_size":(8,8,8),
                                                            "mol_dim":32,
                                                            "feedforward_dim":128,
                                                            "hidden_fusion_dim":128,
                                                            "gnn_hidden_dim":64,
                                                            "gnn_out_dim":64},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:14<00:00,  3.42it/s]


Dataset initialised with 347 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 86/86 [00:22<00:00,  3.81it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7632
ROC AUC=0.7595238095238096
Epoch 1 Done. Average Loss=0.7536
ROC AUC=0.6244047619047619
Epoch 2 Done. Average Loss=0.6718
ROC AUC=0.655952380952381
Epoch 3 Done. Average Loss=0.6587
ROC AUC=0.6482142857142857
Epoch 4 Done. Average Loss=0.6852
ROC AUC=0.7452380952380953
Epoch 5 Done. Average Loss=0.6718
ROC AUC=0.6666666666666666
Early stopping triggered at epoch 5. Best ROC AUC=0.7595238095238096
Average Precision=0.6282


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:50<00:00,  2.31it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7367
ROC AUC=0.6672727272727272
Epoch 1 Done. Average Loss=0.6937
ROC AUC=0.6624242424242424
Epoch 2 Done. Average Loss=0.6744
ROC AUC=0.6775757575757576
Epoch 3 Done. Average Loss=0.6493
ROC AUC=0.686060606060606
Epoch 4 Done. Average Loss=0.6874
ROC AUC=0.6436363636363637
Epoch 5 Done. Average Loss=0.6905
ROC AUC=0.6799999999999999
Epoch 6 Done. Average Loss=0.6757
ROC AUC=0.6842424242424242
Epoch 7 Done. Average Loss=0.6706
ROC AUC=0.6854545454545454
Epoch 8 Done. Average Loss=0.6824
ROC AUC=0.6687878787878788
Early stopping triggered at epoch 8. Best ROC AUC=0.686060606060606
Average Precision=0.5448


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 257/257 [01:01<00:00,  4.19it/s]


Dataset initialised with 348 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 85/85 [00:27<00:00,  3.11it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7513
ROC AUC=0.6672727272727272
Epoch 1 Done. Average Loss=0.6867
ROC AUC=0.7448484848484849
Epoch 2 Done. Average Loss=0.6862
ROC AUC=0.6818181818181818
Epoch 3 Done. Average Loss=0.6770
ROC AUC=0.743030303030303
Epoch 4 Done. Average Loss=0.6732
ROC AUC=0.6503030303030303
Epoch 5 Done. Average Loss=0.6650
ROC AUC=0.6818181818181819
Epoch 6 Done. Average Loss=0.6776
ROC AUC=0.7163636363636364
Early stopping triggered at epoch 6. Best ROC AUC=0.7448484848484849
Average Precision=0.6558
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.692916 |                  0.565139 |                     0.713441 |            0.642938 |          0.53278  | 0.820987 |   0.666621 |

### ConvMixer26KNNGCN with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.692607 |                  0.570078 |                     0.668817 |            0.669968 |          0.569051 | 0.786254 |   0.670007 |
| Both   |  0.683291 |                  0.577093 |                     0.57043  |            0.773214 |          0.604518 | 0.764448 |   0.701915 |

In [9]:
from models.GCNs.ConvMixer26KNNGCNwithCrossAttentionFusion import Model as ConvMixer26KNNGCNwithCAFusion
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixer26KNNGCNwithCAFusion,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "patch_size":(8,8,8),
                                                            "mol_dim":32,
                                                            "feedforward_dim":128,
                                                            "hidden_fusion_dim":128,
                                                            "gnn_hidden_dim":64,
                                                            "gnn_out_dim":64},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [01:15<00:00,  3.39it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:25<00:00,  3.41it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7098
ROC AUC=0.7565476190476191
Epoch 1 Done. Average Loss=0.6922
ROC AUC=0.7386904761904763
Epoch 2 Done. Average Loss=0.6896
ROC AUC=0.7363095238095239
Epoch 3 Done. Average Loss=0.6640
ROC AUC=0.7005952380952382
Epoch 4 Done. Average Loss=0.6630
ROC AUC=0.7401785714285715
Epoch 5 Done. Average Loss=0.6798
ROC AUC=0.6577380952380952
Early stopping triggered at epoch 5. Best ROC AUC=0.7565476190476191
Average Precision=0.6465


100%|████████████████████████████████████████████████████████████████████| 256/256 [02:04<00:00,  2.05it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7394
ROC AUC=0.6909090909090909
Epoch 1 Done. Average Loss=0.7040
ROC AUC=0.7109090909090909
Epoch 2 Done. Average Loss=0.6526
ROC AUC=0.6533333333333333
Epoch 3 Done. Average Loss=0.6869
ROC AUC=0.6527272727272728
Epoch 4 Done. Average Loss=0.6546
ROC AUC=0.6663636363636364
Epoch 5 Done. Average Loss=0.6545
ROC AUC=0.6884848484848485
Epoch 6 Done. Average Loss=0.6759
ROC AUC=0.6503030303030303
Early stopping triggered at epoch 6. Best ROC AUC=0.7109090909090909
Average Precision=0.5910


100%|████████████████████████████████████████████████████████████████████| 257/257 [01:09<00:00,  3.68it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:24<00:00,  3.43it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7349
ROC AUC=0.6987878787878788
Epoch 1 Done. Average Loss=0.6791
ROC AUC=0.6496969696969697
Epoch 2 Done. Average Loss=0.6874
ROC AUC=0.7193939393939394
Epoch 3 Done. Average Loss=0.6792
ROC AUC=0.7127272727272727
Epoch 4 Done. Average Loss=0.6847
ROC AUC=0.7303030303030302
Epoch 5 Done. Average Loss=0.6712
ROC AUC=0.7557575757575757
Epoch 6 Done. Average Loss=0.6644
ROC AUC=0.7139393939393939
Epoch 7 Done. Average Loss=0.6619
ROC AUC=0.5739393939393939
Epoch 8 Done. Average Loss=0.6631
ROC AUC=0.7066666666666667
Epoch 9 Done. Average Loss=0.6547
ROC AUC=0.7
Epoch 10 Done. Average Loss=0.6585
ROC AUC=0.7351515151515151
Early stopping triggered at epoch 10. Best ROC AUC=0.7557575757575757
Average Precision=0.6226
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-------------

### ConvMixer18KNNGCN with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.691443 |                  0.557232 |                     0.620968 |            0.718912 |          0.565422 | 0.781472 |   0.684302 |
| Both   |  0.671671 |                  0.539429 |                     0.801882 |            0.547484 |          0.497376 | 0.830451 |   0.637688 |

In [10]:
from models.GCNs.ConvMixer18KNNGCNwithCrossAttentionFusion import Model as ConvMixer18KNNGCNwithCAFusion
optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixer18KNNGCNwithCAFusion,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "patch_size":(8,8,8),
                                                            "mol_dim":32,
                                                            "feedforward_dim":128,
                                                            "hidden_fusion_dim":128,
                                                            "gnn_hidden_dim":64,
                                                            "gnn_out_dim":64},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|████████████████████████████████████████████████████████████████████| 256/256 [01:25<00:00,  3.00it/s]


Dataset initialised with 347 entries.


100%|██████████████████████████████████████████████████████████████████████| 86/86 [00:30<00:00,  2.78it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.8027
ROC AUC=0.7535714285714287
Epoch 1 Done. Average Loss=0.7086
ROC AUC=0.7208333333333333
Epoch 2 Done. Average Loss=0.6627
ROC AUC=0.6738095238095239
Epoch 3 Done. Average Loss=0.6731
ROC AUC=0.736904761904762
Epoch 4 Done. Average Loss=0.6847
ROC AUC=0.725
Epoch 5 Done. Average Loss=0.6666
ROC AUC=0.7452380952380953
Early stopping triggered at epoch 5. Best ROC AUC=0.7535714285714287
Average Precision=0.6332


100%|████████████████████████████████████████████████████████████████████| 256/256 [02:21<00:00,  1.81it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7646
ROC AUC=0.6642424242424242
Epoch 1 Done. Average Loss=0.6912
ROC AUC=0.6115151515151515
Epoch 2 Done. Average Loss=0.6668
ROC AUC=0.693939393939394
Epoch 3 Done. Average Loss=0.6700
ROC AUC=0.6684848484848485
Epoch 4 Done. Average Loss=0.6735
ROC AUC=0.7096969696969697
Epoch 5 Done. Average Loss=0.6578
ROC AUC=0.6793939393939394
Epoch 6 Done. Average Loss=0.6608
ROC AUC=0.6830303030303031
Epoch 7 Done. Average Loss=0.6732
ROC AUC=0.6866666666666666
Epoch 8 Done. Average Loss=0.6621
ROC AUC=0.68
Epoch 9 Done. Average Loss=0.6602
ROC AUC=0.6781818181818181
Early stopping triggered at epoch 9. Best ROC AUC=0.7096969696969697
Average Precision=0.5918


100%|████████████████████████████████████████████████████████████████████| 257/257 [01:15<00:00,  3.42it/s]


Dataset initialised with 348 entries.


100%|██████████████████████████████████████████████████████████████████████| 85/85 [00:23<00:00,  3.66it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7598
ROC AUC=0.7418181818181818
Epoch 1 Done. Average Loss=0.6894
ROC AUC=0.7593939393939394
Epoch 2 Done. Average Loss=0.6829
ROC AUC=0.7015151515151515
Epoch 3 Done. Average Loss=0.6779
ROC AUC=0.7081818181818182
Epoch 4 Done. Average Loss=0.6675
ROC AUC=0.726060606060606
Epoch 5 Done. Average Loss=0.6575
ROC AUC=0.7563636363636363
Epoch 6 Done. Average Loss=0.6854
ROC AUC=0.7496969696969698
Early stopping triggered at epoch 6. Best ROC AUC=0.7593939393939394
Average Precision=0.6139
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.691443 |                  0.557232 |                     0.620968 |            0.718912 |          0.565422 | 0.781472 |   0.684302 |

### ConvMixerGCNTransformer with Cross Attention Fusion
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------:|-----------------------------:|--------------------:|------------------:|---------:|-----------:|
| preNac |  0.676805 |                  0.501041 |                     0.737903 |            0.547971 |          0.468095 | 0.810449 |   0.614124 |
| Both   |  0.661891 |                  0.495072 |                     0.754301 |            0.534334 |          0.465955 | 0.816236 |   0.611218 |

In [10]:
from models.ConvMixerGCNTransformerWithCAFusion import Model as ConvMixerGCNTransformerwithCAFusion

optimiser_class = lambda params:torch.optim.AdamW(params,lr=best_params["lr"],weight_decay=best_params["weight_decay"])
loss_fn = torch.nn.BCEWithLogitsLoss()
four_fold_scores = four_fold_cv_train(ConvMixerGCNTransformerwithCAFusion,
                                             optimiser_class,
                                             {0:1,1:2},
                                             num_epochs=20,
                                             batch_size=8,
                                             combine_timepoints=False,
                                             output_features=1,
                                             loss_fn=loss_fn,
                                             score_fn=evaluate_roc_auc,
                                             score_name="ROC AUC",
                                             probs_fn=get_scores,
                                             modelKwargs={"dim":128,
                                                            "depth":6,
                                                            "kernel_size":(1,9,9),
                                                            "image_size":(16,128,128),
                                                            "patch_size":(8,8,8),
                                                            "gnn_hidden_dim":64,
                                                            "gnn_out_dim":64,
                                                            "num_heads":4,
                                                            "feedforward_dim":128,
                                                            "num_blocks":6,
                                                            "mol_dim":32,
                                                            "hidden_fusion_dim":128,},
                                             trainKwargs={"mode":"both"},
                                             testKwargs={"mode":"preNac"})
print(four_fold_scores)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:04<00:00,  3.99it/s]


Dataset initialised with 347 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 86/86 [00:24<00:00,  3.57it/s]


Dataset initialised with 86 entries.
Epoch 0 Done. Average Loss=0.7661
ROC AUC=0.7360119047619048
Epoch 1 Done. Average Loss=0.7183
ROC AUC=0.7419642857142856
Epoch 2 Done. Average Loss=0.6740
ROC AUC=0.7297619047619048
Epoch 3 Done. Average Loss=0.6712
ROC AUC=0.7446428571428572
Epoch 4 Done. Average Loss=0.6708
ROC AUC=0.7398809523809524
Epoch 5 Done. Average Loss=0.6719
ROC AUC=0.7547619047619047
Epoch 6 Done. Average Loss=0.6746
ROC AUC=0.7273809523809522
Epoch 7 Done. Average Loss=0.6641
ROC AUC=0.7529761904761905
Epoch 8 Done. Average Loss=0.6726
ROC AUC=0.7386904761904762
Epoch 9 Done. Average Loss=0.6630
ROC AUC=0.7678571428571428
Epoch 10 Done. Average Loss=0.6655
ROC AUC=0.7386904761904762
Epoch 11 Done. Average Loss=0.6693
ROC AUC=0.7386904761904762
Epoch 12 Done. Average Loss=0.6700
ROC AUC=0.7345238095238095
Epoch 13 Done. Average Loss=0.6650
ROC AUC=0.7386904761904762
Epoch 14 Done. Average Loss=0.6660
ROC AUC=0.7386904761904762
Early stopping triggered at epoch 14. Best 

100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [01:48<00:00,  2.35it/s]


...

Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7710
ROC AUC=0.7233333333333334
Epoch 1 Done. Average Loss=0.7059
ROC AUC=0.6736363636363636
Epoch 2 Done. Average Loss=0.6878
ROC AUC=0.6863636363636364
Epoch 3 Done. Average Loss=0.6677
ROC AUC=0.7024242424242424
Epoch 4 Done. Average Loss=0.6633
ROC AUC=0.6793939393939393
Epoch 5 Done. Average Loss=0.6668
ROC AUC=0.6806060606060607
Early stopping triggered at epoch 5. Best ROC AUC=0.7233333333333334
Average Precision=0.5187


100%|██████████████████████████████████████████████████████████████████████████████████████████████████| 257/257 [00:59<00:00,  4.28it/s]


Dataset initialised with 348 entries.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 85/85 [00:18<00:00,  4.56it/s]


Dataset initialised with 85 entries.
Epoch 0 Done. Average Loss=0.7362
ROC AUC=0.7196969696969697
Epoch 1 Done. Average Loss=0.6905
ROC AUC=0.7193939393939395
Epoch 2 Done. Average Loss=0.6582
ROC AUC=0.7254545454545455
Epoch 3 Done. Average Loss=0.7042
ROC AUC=0.7196969696969697
Epoch 4 Done. Average Loss=0.6606
ROC AUC=0.7172727272727273
Epoch 5 Done. Average Loss=0.6747
ROC AUC=0.7257575757575758
Epoch 6 Done. Average Loss=0.6625
ROC AUC=0.6942424242424243
Epoch 7 Done. Average Loss=0.6643
ROC AUC=0.7196969696969697
Epoch 8 Done. Average Loss=0.6775
ROC AUC=0.7196969696969697
Epoch 9 Done. Average Loss=0.6700
ROC AUC=0.6903030303030303
Epoch 10 Done. Average Loss=0.6697
ROC AUC=0.7193939393939395
Early stopping triggered at epoch 10. Best ROC AUC=0.7257575757575758
Average Precision=0.5454
|        |   ROC AUC |   Average Precision Score |   Sensitivity / Recall / TPR |   Specificity / TNR |   PPV / Precision |      NPV |   Accuracy |
|:-------|----------:|--------------------------